In [19]:
from langchain_openai import ChatOpenAI
import os

model = ChatOpenAI(
    model_name="qwen-max",
    openai_api_key = os.getenv("DASHSCOPE_API_KEY"),
    openai_api_base = "https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.9, 
    max_tokens=5000,
    verbose=True
)

In [20]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

output_parser = StrOutputParser()

In [21]:
# 初始论点
planner = (
    ChatPromptTemplate.from_template("生成关于以下内容的论点: {input}")
    | model
    | output_parser
    | {"base_response": RunnablePassthrough()}
)

In [22]:
# 正方意见
argument_for = (
    ChatPromptTemplate.from_template("列出关于{base_response}的正面或有利的方面")
    | model
    | output_parser
)

In [23]:
# 反方意见
argument_against = (
    ChatPromptTemplate.from_template("列出关于{base_response}的反面或不利的方面")
    | model
    | output_parser
)

In [24]:
# 总结陈词
final_responder = (
    ChatPromptTemplate.from_messages([
        ("ai", "{original_response}"),
        ("human", "正面观点:\n{result_1}\n\n反面观点:\n{result_2}"),
        ("system", "给出评价后生成最终回应")
    ])
    | model
    | output_parser
)

In [25]:
# 处理链
chain = (
    planner
    | {
        "result_1": argument_for,
        "result_2": argument_against,
        "original_response": itemgetter("base_response"),
    }
    | final_responder
)

In [26]:
print(chain.invoke({"input": "房地产低迷"}, verbose=True))

### 评价与最终回应

#### 正面观点的评价
正面观点强调了房地产市场低迷可能带来的积极影响，这些观点有助于我们更全面地理解市场调整的多方面作用。具体来说：

- **促进资源优化配置** 和 **为未来增长奠定基础**：通过市场调整，低效企业被淘汰，资源更加集中于优质企业，从而提高了整个行业的竞争力。
- **保障社会稳定与公平** 和 **引导长期健康发展**：政府调控措施可以抑制投机行为，确保市场的健康和稳定，支持普通家庭实现住房梦想。
- **推动产品创新和服务改进** 和 **促进城市规划科学化**：供需关系变化促使开发商创新产品和服务，同时也有助于城市规划的优化。
- **增强金融机构风险控制能力** 和 **鼓励多元化投资渠道**：金融环境收紧使得金融机构更加审慎，投资者也寻找更多元化的投资机会。
- **培养理性消费观念** 和 **加速去杠杆化进程**：市场不确定性促使消费者和投资者更加理性，减少过度负债。
- **激发新兴产业机遇** 和 **提高生活品质**：技术进步和新生活方式带来了新的商业机会，提升了人们的生活质量。

#### 反面观点的评价
反面观点则关注房地产市场低迷可能带来的不利影响，这些观点提醒我们需要警惕潜在的风险和挑战。具体来说：

- **长期负面影响** 和 **社会问题**：持续的市场低迷可能导致经济衰退、失业率上升和社会不稳定。
- **过度干预风险** 和 **地方财政压力**：政府调控措施不当可能会导致市场流动性不足，影响地方政府收入。
- **资源错配** 和 **城镇化进程放缓带来的挑战**：供需失衡可能导致结构性过剩或短缺，城镇化速度减慢也会带来转型压力。
- **融资成本提高** 和 **消费者负担加重**：金融环境收紧增加了开发商和购房者的财务压力。
- **负面反馈循环** 和 **资本外流**：悲观情绪可能形成自我实现预言，资金撤离不利于市场稳定。
- **传统区域价值下降** 和 **适应性难题**：新技术和新生活方式的变化可能使一些传统区域的价值下降，老旧项目面临淘汰风险。

### 最终回应

综上所述，房地产市场的低迷是一个复杂的现象，既有其积极的一面，也有潜在的风险。为了更好地应对这一挑战，建议采取以下策略：

1. **平衡调控力度**：政府应灵活调整政策，避免过度干预市场，同时确保